# In-Play Win Probability Predictor
This notebook predicts the probability of the chasing team winning at any point during the second innings.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data & Find 1st Innings Targets

In [2]:
matches = pd.read_csv('data/interim/matches_clean.csv')
deliveries = pd.read_csv('data/interim/deliveries_clean.csv')

# Get total score of 1st innings to set the target for 2nd innings
total_score_df = deliveries.groupby(['match_id', 'inning']).sum()['total_runs'].reset_index()
total_score_df = total_score_df[total_score_df['inning'] == 1]
total_score_df['target'] = total_score_df['total_runs'] + 1

# Merge the target with the matches dataframe
match_df = matches.merge(total_score_df[['match_id', 'target']], left_on='id', right_on='match_id')

## 2. Feature Engineering for 2nd Innings

In [3]:
# Filter deliveries for 2nd innings only
delivery_df = match_df[['match_id', 'city', 'winner', 'target']].merge(deliveries, on='match_id')
delivery_df = delivery_df[delivery_df['inning'] == 2]

# Calculate running cumulative score
delivery_df['current_score'] = delivery_df.groupby('match_id')['total_runs'].cumsum()

# Calculate runs left
delivery_df['runs_left'] = delivery_df['target'] - delivery_df['current_score']

# Calculate balls left
delivery_df['balls_left'] = 120 - (delivery_df['over'] - 1) * 6 - delivery_df['ball']
# Ensure we don't have negative balls left (can happen if extras are bowled at the end)
delivery_df['balls_left'] = delivery_df['balls_left'].apply(lambda x: x if x >= 0 else 0)

# Calculate wickets left
# Create a 1/0 flag for a dismissal
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].fillna('0')
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].apply(lambda x: x if x == '0' else '1')
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].astype(int)

# Cumulative sum of wickets
wickets_fallen = delivery_df.groupby('match_id')['player_dismissed'].cumsum().values
delivery_df['wickets_left'] = 10 - wickets_fallen

# Calculate CRR and RRR
delivery_df['crr'] = (delivery_df['current_score'] * 6) / (120 - delivery_df['balls_left'])
delivery_df['rrr'] = (delivery_df['runs_left'] * 6) / delivery_df['balls_left']

# Did the batting team win? (Label)
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0
delivery_df['result'] = delivery_df.apply(result, axis=1)

final_df = delivery_df[['batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 'wickets_left', 'target', 'crr', 'rrr', 'result']]
final_df = final_df.dropna() # Drop any remaining nulls or infs (e.g., from balls_left=0)
final_df = final_df[final_df['balls_left'] != 0] # Remove last ball to avoid inf RRR


## 3. Train/Test Split & Encoding

In [4]:
# Prepare Features and Label
X = final_df.drop('result', axis=1)
y = final_df['result']

# One-hot encode teams and cities
X_encoded = pd.get_dummies(X, columns=['batting_team', 'bowling_team', 'city'])

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
print(f"Total training balls: {len(X_train)}")

Total training balls: 57634


## 4. Train the Logistic Regression Model

In [5]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"Accuracy on unseen deliveries: {accuracy_score(y_test, y_pred)*100:.2f}%")

Accuracy on unseen deliveries: 82.32%


## 5. Win Probability Predictor Function

In [6]:
def predict_win_probability(batting_team, bowling_team, city, target, current_score, overs_completed, wickets_fallen):
    runs_left = target - current_score
    balls_left = 120 - (overs_completed * 6)
    wickets_left = 10 - wickets_fallen
    crr = current_score / overs_completed if overs_completed > 0 else 0
    rrr = (runs_left * 6) / balls_left if balls_left > 0 else 0
    
    # Create input dataframe
    input_data = pd.DataFrame({
        'runs_left': [runs_left],
        'balls_left': [balls_left],
        'wickets_left': [wickets_left],
        'target': [target],
        'crr': [crr],
        'rrr': [rrr]
    })
    
    # Create dummy columns to match training data
    for col in X_encoded.columns:
        if col not in input_data.columns:
            if f"batting_team_{batting_team}" == col or f"bowling_team_{bowling_team}" == col or f"city_{city}" == col:
                input_data[col] = 1
            else:
                input_data[col] = 0
                
    # Ensure column order matches X_encoded exactly
    input_data = input_data[X_encoded.columns]
                
    # Predict probabilities
    prob = model.predict_proba(input_data)[0]
    
    print(f"--- MATCH SCENARIO ---")
    print(f"{batting_team} chasing {target} against {bowling_team} in {city}")
    print(f"Score: {current_score}/{wickets_fallen} in {overs_completed} overs")
    print(f"Runs needed: {runs_left} off {balls_left} balls")
    print(f"----------------------")
    print(f"Win Probability for {batting_team}: {prob[1]*100:.1f}%")
    print(f"Win Probability for {bowling_team}: {prob[0]*100:.1f}%")

# Let's test a hypothetical scenario:
# RCB chasing 180 against CSK in Bangalore. They are 100/2 after 12 overs.
predict_win_probability(
    batting_team='RCB', 
    bowling_team='CSK', 
    city='Bangalore', 
    target=180, 
    current_score=100, 
    overs_completed=12, 
    wickets_fallen=2
)

--- MATCH SCENARIO ---
RCB chasing 180 against CSK in Bangalore
Score: 100/2 in 12 overs
Runs needed: 80 off 48 balls
----------------------
Win Probability for RCB: 78.5%
Win Probability for CSK: 21.5%
